# Setup

In [1]:
import faulthandler
faulthandler.enable()

from pathlib import Path


from fjsspw_solver.mutations import SequenceSwapMutation, MachineSwapMutation, WorkerSwapMutation
from fjsspw_solver import Individual, Encoding, print_address
from fjsspw_solver.operators import CrossoverParams, MutationParams
from fjsspw_solver.crossovers import POXCrossover


from util.benchmark_parser import WorkerBenchmarkParser

from constants import *


In [2]:
parser = WorkerBenchmarkParser()
instance_name = 'Fattahi5'
instance_path = INSTANCE_FJSSPW_PATH / f'{instance_name}.fjs'
encoding = parser.parse_benchmark(str(instance_path))

encoding = Encoding(encoding.durations(), encoding.job_sequence())

In [3]:
indv1 = Individual(encoding)
indv1.random_init()
indv2 = Individual(encoding)
indv2.random_init()

In [4]:
from fjsspw_solver.operators import Crossovers, Mutations


gene_mutation_params = MutationParams(
    mutation=Mutations.GENE_SWAP,
    mutation_prob=1.0,
)
machine_mutation_params = MutationParams(
    mutation=Mutations.MACHINE_SWAP,
    mutation_prob=1.0,
)
worker_mutation_params = MutationParams(
    mutation=Mutations.WORKER_SWAP,
    mutation_prob=1.0,
)

pox_crossover_params = CrossoverParams(
    crossover=Crossovers.POX,
    crossover_prob=1.0,
)
mox_crossover_params = CrossoverParams(
    crossover=Crossovers.MOX,
    crossover_prob=1.0,
)

In [5]:
from fjsspw_solver.crossovers import get_crossover
from fjsspw_solver.mutations import get_mutation


mutation_op = get_mutation(worker_mutation_params, encoding)
crossover_op = get_crossover(pox_crossover_params, encoding)

In [6]:
print("n_jobs:", encoding.n_jobs())
print("n_operations:", encoding.n_operations())
print("n_workers:", encoding.n_workers())
print("n_machines:", encoding.n_machines())

n_jobs: 3
n_operations: 6
n_workers: 3
n_machines: 2


# Test mutation

In [7]:
print("Before:", indv1.to_string())
mutation_op.mutate(indv1)
mutation_op.mutate(indv1)
mutation_op.mutate(indv1)
mutation_op.mutate(indv1)
mutation_op.mutate(indv1)
print(" After:", indv1.to_string())

Before: Individual(sequence={1, 0, 2, 1, 2, 0}, machines={1, 1, 1, 1, 0, 1}, workers={2, 2, 2, 1, 0, 1})
 After: Individual(sequence={1, 0, 2, 1, 2, 0}, machines={1, 1, 1, 1, 0, 1}, workers={2, 1, 2, 1, 2, 0})


# Test crossover

In [83]:
# crossover_op = get_crossover(pox_crossover_params, encoding)
crossover_op = get_crossover(mox_crossover_params, encoding)

print("Before:")
print(f"{indv1.to_string()}, fitness: {indv1.get_internal_fitness()}")
print(f"{indv2.to_string()}, fitness: {indv2.get_internal_fitness()}")

offspring1, offspring2 = crossover_op.crossover(indv1, indv2)

print("After:")
print(f"{offspring1.to_string()}, fitness: {offspring1.get_internal_fitness()}")
print(f"{offspring2.to_string()}, fitness: {offspring2.get_internal_fitness()}")

Before:
Individual(sequence={1, 0, 2, 1, 2, 0}, machines={1, 1, 1, 1, 0, 1}, workers={2, 1, 2, 1, 2, 0}), fitness: 211
Individual(sequence={1, 2, 0, 1, 2, 0}, machines={0, 1, 1, 0, 0, 1}, workers={0, 0, 1, 2, 1, 1}), fitness: 186
After:
Individual(sequence={1, 2, 0, 1, 2, 0}, machines={0, 1, 1, 0, 0, 1}, workers={0, 1, 1, 2, 1, 1}), fitness: 177
Individual(sequence={1, 0, 2, 1, 2, 0}, machines={1, 1, 1, 1, 0, 1}, workers={2, 0, 2, 1, 2, 0}), fitness: 220
